### Runing LLaMA-2-13B W2A16 quantized model

#### Download the prebuilt quantized model:
We have provide the prebuilt quantized model on Huggingface. In order to download the large weights, we'll have to use git lfs.

In [ ]:
!git lfs install

# download LLaMA-2-13b-w2a16 quantization
!git clone https://huggingface.co/FRM-PTQ/llama-2-13b-w2a16-frm-ptq

In [6]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [7]:
from accelerate import infer_auto_device_map, dispatch_model
import torch
from datautils import get_loaders, test_ppl

@torch.no_grad()
def evaluate(model, tokenizer):
    '''
    Note: evaluation simply move model to single GPU. 
    Therefor, to evaluate large model such as Llama-2-70B on single A100-80GB,
    please activate '--real_quant'.
    '''
    # import pdb;pdb.set_trace()
    block_class_name = model.model.layers[0].__class__.__name__
    device_map = infer_auto_device_map(model, max_memory={i: '40GB' for i in range(torch.cuda.device_count())}, no_split_module_classes=[block_class_name])
    model = dispatch_model(model, device_map=device_map)
    results = {}

    datasets = ["wikitext2", "c4"]
    ppl_results = test_ppl(model, tokenizer, datasets, 2048)
    for dataset in ppl_results:
        print(f'{dataset} perplexity: {ppl_results[dataset]:.2f}')
    return results

In [ ]:
from quantize.int_linear_real import load_quantized_model
from accelerate import infer_auto_device_map, dispatch_model
import torch

model_path = './llama-2-13b-w2g128'
wbits = 2
abits = 16
group_size = 128
use_act_quant = False

model, tokenizer = load_quantized_model(model_path=model_path, wbits=wbits, abits=abits, group_size=group_size, use_act_quant=use_act_quant)
model = model.cuda()
# Test PPL
evaluate(model, tokenizer)

Loading quantized model from /root/code_z/EfficientQAT/output/block_ap_models/llama-2-13b-w2g128


100%|██████████| 40/40 [00:00<00:00, 74.65it/s]


Loading pre-computed quantized weights...
Loading pre-computed quantized weights Successfully
get_wikitext2


Using the latest cached version of the module from /root/.cache/huggingface/modules/datasets_modules/datasets/wikitext/a241db52902eaf2c6aa732210bead40c090019a499ceb13bcbfa3f8ab646a126 (last modified on Thu Nov 23 17:36:43 2023) since it couldn't be found locally at wikitext, or remotely on the Hugging Face Hub.
Using the latest cached version of the module from /root/.cache/huggingface/modules/datasets_modules/datasets/wikitext/a241db52902eaf2c6aa732210bead40c090019a499ceb13bcbfa3f8ab646a126 (last modified on Thu Nov 23 17:36:43 2023) since it couldn't be found locally at wikitext, or remotely on the Hugging Face Hub.
100%|██████████| 166/166 [01:17<00:00,  2.13it/s]


wikitext2:7.127840995788574
get_c4


100%|██████████| 256/256 [02:00<00:00,  2.12it/s]

c4:9.418624877929688
wikitext2 perplexity: 7.13
c4 perplexity: 9.42


{}

In [ ]:
# Test Zero_shot
import lm_eval
from lm_eval.models.huggingface import HFLM
from lm_eval.utils import make_table
eval_tasks = 'piqa,arc_easy,arc_challenge,hellaswag,boolq,winogrande,mmlu'
task_list = eval_tasks.split(',')
model = HFLM(pretrained=model, batch_size=8)
task_manager = lm_eval.tasks.TaskManager()
results = lm_eval.simple_evaluate(
        model=model,
        tasks=task_list,
        num_fewshot=0,
        task_manager=task_manager,
        )
print(make_table(results))
total_acc = 0
for task in task_list:
    total_acc += results['results'][task]['acc,none']
print(f'Average Acc: {total_acc/len(task_list)*100:.2f}%')